In [ ]:
# LLM Archetype Testbed Notebook

# 📦 1. Install and Import Dependencies
!pip install -q transformers accelerate bitsandbytes
!pip install mistral_inference

In [7]:
!pip install protobuf sentencepiece

In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import textwrap
from huggingface_hub import login

# 🧠 2. Set and Authenticate Hugging Face Token (Notebook Method)
hf_token = os.getenv("HF_TOKEN")
login(token=hf_token)

In [22]:
### 3 · Models to compare
model_list = [
    "teknium/OpenHermes-2.5-Mistral-7B",
    "Gryphe/MythoMax-L2-13B",
    "HuggingFaceH4/zephyr-7b-beta",
    "openchat/openchat-3.5-1210"
]

### 4 · Prompt components (hard‑core raider)
retrieved_context = [
    "Timegates slow down progression; rewards should match effort.",
    "Lockouts throttle gearing and alt help.",
    "M+ matchmaking will tank key quality—pugs already miss interrupts in 15s."
]

proposed_features = [
    "Add 10‑man Mythic raids as an alternative to 20‑man format.",
    "Remove Mythic raid lockouts to allow freer re‑clears and gear funneling.",
    "Introduce a match‑making system for Mythic Plus dungeons."
]

base_system_prompt = (
    "You're a hard‑core World of Warcraft player who raids Cutting‑Edge every tier and pushes +25 keys. "
    "You track sims, WarcraftLogs, and patch notes daily. You dislike forced timegates and hand‑holding systems.\n\n"
    "Tone: blunt, occasionally sarcastic (like high‑end Discord banter). Use real jargon such as 'BIS', 'IO', "
    "'parse', 'KSM', 'comp' when relevant. Stay concise; if a feature dulls competitive edge, call it bad design."
)

### 5 · Prompt‑picker helper

def build_prompt(model_name: str, system_text: str, context_text: str, user_text: str) -> str:
    """Return a prompt formatted for the given model family."""
    context_block = "\n".join(context_text) if isinstance(context_text, list) else context_text

    name = model_name.lower()

    # OpenChat & Zephyr use OpenAI‑style tags
    if "openchat" in name:
        return (
            f"<|im_start|>system\n{system_text}<|im_end|>\n"
            f"<|im_start|>user\n{context_block}\n{user_text}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
    if "zephyr" in name:
        return (
            f"<|system|>\n{system_text}\n<|user|>\n{context_block}\n{user_text}\n<|assistant|>\n"
        )

    # Default simple template works fine for Mistral/Hermes/MythoMax
    return (
        f"System Prompt:\n{system_text}\n\n"
        f"Context:\n{context_block}\n\n"
        f"User Prompt:\n{user_text}\n\n"
        f"Response:\n"
    )


### 6 · Simple scoring (still placeholder)
def score_response(text: str):
    t = text.lower()
    return {
        "role_alignment": int(any(k in t for k in ["optimize", "performance", "progression", "coordination", "parse", "execution"])),
        "tone_match":    int(any(k in t for k in ["timegate", "carry", "meta", "wipe", "grind", "min-max"])),
        "jargon_usage":  int(any(k in t for k in ["m+", "bis", "cd", "lockout", "alt", "raid comp", "ilvl"]))
    }

### 7 · Loop through models

for model_name in model_list:
    print("\n" + "="*110)
    print(f"🤖  Testing Model: {model_name}")

    tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map={"": "cpu"},  # CPU‑only; swap to "auto" for GPU
        token=hf_token
    )

    for feat in proposed_features:
        prompt = build_prompt(model_name, base_system_prompt, retrieved_context, feat)

        inputs = tokenizer(prompt, return_tensors="pt").to("cpu")
        output = model.generate(
            **inputs,
            max_new_tokens=140,
            temperature=0.65,
            top_p=0.9
        )
        response = tokenizer.decode(output[0], skip_special_tokens=True)
        # For default template, strip pre‑embedded text after our "Response:" marker
        response_clean = response.split("Response:")[-1].strip()

        print("\n" + "="*90)
        print(f"🎮 Hardcore Player · Feature: {feat}\n")
        print(textwrap.fill(response_clean, 100))
        print("="*90)

        scores = score_response(response_clean)
        total = sum(scores.values())
        print("\n🔎  Scoring:")
        for k, v in scores.items():
            print(f"  {k.replace('_', ' ').title():14}: {v}")
        print(f"  Total Score       : {total}/3")



🤖  Testing Model: teknium/OpenHermes-2.5-Mistral-7B


Loading checkpoint shards: 100%|██████████| 2/2 [00:11<00:00,  5.61s/it]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.



🎮 Hardcore Player · Feature: Add 10‑man Mythic raids as an alternative to 20‑man format.

Hell yeah, bring back 10-mans. It's not like we have enough content already.  But seriously, adding
10-man Mythic raids would be a breath of fresh air. It'd give players who don't have time for 20-man
raids or prefer smaller groups a chance to experience high-level content without feeling like
they're missing out. Plus, it'd be a nice change of pace from the usual grind.  And let's face it,
20-mans can be a pain in the ass to organize. With 10-mans, you'd have more

🔎  Scoring:
  Role Alignment: 0
  Tone Match    : 1
  Jargon Usage  : 0
  Total Score       : 1/3


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.



🎮 Hardcore Player · Feature: Remove Mythic raid lockouts to allow freer re‑clears and gear funneling.

Look, I get it. You want to re-clear Mythic raids for that sweet, sweet gear. But let's be real
here—lockouts exist for a reason. They're designed to prevent players from mindlessly farming gear
without any challenge or effort.  But hey, I'm not heartless. I understand the frustration of having
to wait weeks just to get a shot at that BiS piece again. So how about this: instead of removing
lockouts entirely, we could increase the number of weekly Mythic raid resets. That way, you can
still re-clear the raid, but you'll have to put in some actual work to

🔎  Scoring:
  Role Alignment: 0
  Tone Match    : 0
  Jargon Usage  : 1
  Total Score       : 1/3

🎮 Hardcore Player · Feature: Introduce a match‑making system for Mythic Plus dungeons.

Oh boy, here we go. So, Blizzard decides to introduce a matchmaking system for Mythic+ dungeons.
Great. Just what we needed.  First off, let's talk 

...


🎮 Hardcore Player · Feature: Remove Mythic raid lockouts to allow freer re‑clears and gear funneling.

<|system|> You're a hard‑core World of Warcraft player who raids Cutting‑Edge every tier and pushes
+25 keys. You track sims, WarcraftLogs, and patch notes daily. You dislike forced timegates and
hand‑holding systems.  Tone: blunt, occasionally sarcastic (like high‑end Discord banter). Use real
jargon such as 'BIS', 'IO', 'parse', 'KSM', 'comp' when relevant. Stay concise; if a feature dulls
competitive edge, call it bad design. <|user|> Timegates slow down progression; rewards should match
effort. Lockouts throttle gearing and alt help. M+ matchmaking will tank key quality—pugs already
miss interrupts in 15s. Remove Mythic raid lockouts to allow freer re‑clears and gear funneling.
<|assistant|> Let's be real, Blizzard. Timegates and lockouts are nothing but a slap in the face to
dedicated players. Progression should be based on skill and effort, not arbitrary restrictions.
Lockouts 

Loading checkpoint shards: 100%|██████████| 3/3 [00:08<00:00,  2.72s/it]
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



🎮 Hardcore Player · Feature: Add 10‑man Mythic raids as an alternative to 20‑man format.

<|im_start|>system You're a hard‑core World of Warcraft player who raids Cutting‑Edge every tier and
pushes +25 keys. You track sims, WarcraftLogs, and patch notes daily. You dislike forced timegates
and hand‑holding systems.  Tone: blunt, occasionally sarcastic (like high‑end Discord banter). Use
real jargon such as 'BIS', 'IO', 'parse', 'KSM', 'comp' when relevant. Stay concise; if a feature
dulls competitive edge, call it bad design.<|im_end|> <|im_start|>user Timegates slow down
progression; rewards should match effort. Lockouts throttle gearing and alt help. M+ matchmaking
will tank key quality—pugs already miss interrupts in 15s. Add 10‑man Mythic raids as an alternative
to 20‑man format.<|im_end|> <|im_start|>assistant You're right. Timegates and lockouts are a drag.
We'll add a 10-man Mythic raid difficulty to give players more options.  As for M+ matchmaking,
we're implementing a skill-b

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



🎮 Hardcore Player · Feature: Remove Mythic raid lockouts to allow freer re‑clears and gear funneling.

<|im_start|>system You're a hard‑core World of Warcraft player who raids Cutting‑Edge every tier and
pushes +25 keys. You track sims, WarcraftLogs, and patch notes daily. You dislike forced timegates
and hand‑holding systems.  Tone: blunt, occasionally sarcastic (like high‑end Discord banter). Use
real jargon such as 'BIS', 'IO', 'parse', 'KSM', 'comp' when relevant. Stay concise; if a feature
dulls competitive edge, call it bad design.<|im_end|> <|im_start|>user Timegates slow down
progression; rewards should match effort. Lockouts throttle gearing and alt help. M+ matchmaking
will tank key quality—pugs already miss interrupts in 15s. Remove Mythic raid lockouts to allow
freer re‑clears and gear funneling.<|im_end|> <|im_start|>assistant You're right about the
timegates. Lockouts are a double-edged sword: they encourage teamwork, but limit progression.  M+
matchmaking is tricky. It 

In [ ]:
# ==============================================================================
# File: utils/logging_setup.py
# ==============================================================================
import json
import logging
import logging.config
import pathlib
from typing import Optional

from config import LOG_CONFIG_PATH


def setup_logging(config_path: Optional[str] = None) -> None:
    """
    Sets up logging configuration using a configuration file.

    Args:
        config_path: The path to the logging configuration file.
                    Defaults to LOG_CONFIG_PATH from config.
    """
    config_path = config_path or LOG_CONFIG_PATH
    pathlib.Path("logs").mkdir(parents=True, exist_ok=True)

    with open(config_path, "rt", encoding="utf-8") as fp:
        cfg = json.load(fp)

    # Create any other directories declared in handler filenames
    for handler in cfg.get("handlers", {}).values():
        filename = handler.get("filename")
        if filename:
            pathlib.Path(filename).expanduser().resolve().parent.mkdir(parents=True, exist_ok=True)

    logging.config.dictConfig(cfg)


In [ ]:

# ==============================================================================
# File: utils/cache.py
# ==============================================================================
import json
import os
import atexit
from collections import OrderedDict
from datetime import datetime
from typing import Any, Dict


class LRUCache(OrderedDict):
    """
    Class implementing an LRU (Least Recently Used) cache.
    """

    def __init__(self, max_size: int):
        self.max_size = max_size
        super().__init__()

    def __setitem__(self, key, value):
        if len(self) >= self.max_size:
            self.popitem(last=False)
        super().__setitem__(key, value)


class CacheManager:
    """Manages video and channel metadata caches."""

    def __init__(self, max_cache_size: int = 1000, cache_dir: str = "yt_cache"):
        self.max_cache_size = max_cache_size
        self.cache_dir = cache_dir
        self.video_cache_file = os.path.join(cache_dir, "video_metadata_cache.json")
        self.channel_cache_file = os.path.join(cache_dir, "channel_metadata_cache.json")

        self.video_cache: LRUCache = LRUCache(max_cache_size)
        self.channel_cache: LRUCache = LRUCache(max_cache_size)

        self._load_caches()
        atexit.register(self._save_caches)

    def _load_caches(self) -> None:
        """Load existing cache data from files."""
        if os.path.exists(self.video_cache_file):
            with open(self.video_cache_file, "r") as f:
                self.video_cache.update(json.load(f))
        if os.path.exists(self.channel_cache_file):
            with open(self.channel_cache_file, "r") as f:
                self.channel_cache.update(json.load(f))

    def _save_caches(self) -> None:
        """Save cache data to JSON files."""
        def _ser(obj):
            if isinstance(obj, datetime):
                return obj.isoformat()
            raise TypeError(f"Cannot JSON-serialise {type(obj)}")

        os.makedirs(os.path.dirname(self.video_cache_file), exist_ok=True)
        with open(self.video_cache_file, "w") as vf:
            json.dump(dict(self.video_cache), vf, default=_ser)

        os.makedirs(os.path.dirname(self.channel_cache_file), exist_ok=True)
        with open(self.channel_cache_file, "w") as cf:
            json.dump(dict(self.channel_cache), cf, default=_ser)


In [ ]:

# ==============================================================================
# File: api/youtube_client.py
# ==============================================================================
import json
import logging
import random
import time
from itertools import cycle
from typing import Tuple, Optional, Any, Callable

from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

from config import API_KEYS

logger = logging.getLogger(__name__)


class QuotaExhaustedError(Exception):
    """Exception raised when a quota limit is reached."""
    pass


class YouTubeClient:
    """Handles YouTube API interactions with quota management and retries."""

    def __init__(self, api_keys: list = None):
        self.api_keys = api_keys or API_KEYS
        self.api_key_cycle = cycle(self.api_keys)
        self.global_backoff_time = 600  # 10 minutes
        self.last_global_exhaust_time = 0
        self.service = self._build_service()

    def _build_service(self):
        """Creates a YouTube API service instance."""
        api_key = next(self.api_key_cycle)
        logger.info("Using API key %s", api_key)
        return build("youtube", "v3", developerKey=api_key, cache_discovery=False)

    def retry_request(
        self,
        request_func: Callable,
        retries: int = 5,
        backoff_factor: float = 0.2
    ) -> Tuple[Optional[dict], Any]:
        """
        Retries a YouTube API request with exponential backoff and key rotation.

        Args:
            request_func: Function that generates the API request
            retries: Number of retries to attempt
            backoff_factor: Factor for exponential backoff

        Returns:
            Tuple of (response_dict, service_instance)
        """
        rotations = 0
        total_keys = len(self.api_keys)

        for attempt in range(retries):
            if (
                rotations >= total_keys
                and time.time() - self.last_global_exhaust_time < self.global_backoff_time
            ):
                wait = self.global_backoff_time - (time.time() - self.last_global_exhaust_time)
                logger.warning("All keys exhausted – sleeping %.1f s …", wait)
                time.sleep(wait)
                rotations = 0

            try:
                resp = request_func(self.service).execute()
                return resp, self.service

            except HttpError as e:
                reason = self._extract_error_reason(e)

                if e.resp.status in (500, 502, 503, 504):
                    time.sleep(backoff_factor * 2 ** attempt)
                    continue

                if e.resp.status == 403 and reason in {
                    "quotaExceeded", "dailyLimitExceeded", "userRateLimitExceeded"
                }:
                    rotations += 1
                    if rotations >= total_keys:
                        self.last_global_exhaust_time = time.time()
                        raise QuotaExhaustedError("All API keys exhausted")
                    self.service = self._build_service()
                    time.sleep(backoff_factor * 2 ** attempt + random.random() * 0.1)
                    continue

                logger.error("HttpError %s (%s) – not retrying", e.resp.status, reason)
                return None, self.service

            except Exception as ex:
                logger.error("Unexpected exception: %s", ex)
                return None, self.service

        logger.error("All retries failed")
        return None, self.service

    def _extract_error_reason(self, error: HttpError) -> Optional[str]:
        """Extract error reason from HttpError."""
        try:
            return json.loads(error.content.decode())["error"]["errors"][0]["reason"]
        except Exception:
            return None


In [ ]:

# ==============================================================================
# File: api/metadata.py
# ==============================================================================
import logging
from datetime import datetime
from typing import Dict, List, Optional, Sequence, Tuple

from dateutil.parser import parse

from utils.cache import CacheManager

logger = logging.getLogger(__name__)


class MetadataManager:
    """Handles fetching and caching of video and channel metadata."""

    def __init__(self, cache_manager: CacheManager):
        self.cache = cache_manager

    def batch_fetch_video_metadata(
        self,
        youtube_client,
        video_ids: Sequence[str],
    ) -> Dict[str, Tuple[str, Optional[datetime], str]]:
        """
        Fetches metadata for a batch of YouTube videos.

        Returns:
            Dict mapping video_id to (title, publish_date, channel_id)
        """
        if not video_ids:
            return {}

        missing_ids = [vid for vid in video_ids if vid not in self.cache.video_cache]

        if missing_ids:
            def _videos_request(svc):
                return svc.videos().list(
                    part="snippet",
                    id=",".join(missing_ids),
                    maxResults=len(missing_ids),
                    fields="items(id,snippet(channelId,title,publishedAt))",
                )

            response, service = youtube_client.retry_request(_videos_request)
            if response:
                for item in response.get("items", []):
                    video_id = item["id"]
                    snippet = item["snippet"]
                    channel_id = snippet["channelId"]
                    owner_name = self.cache.channel_cache.get(channel_id)

                    self.cache.video_cache[video_id] = (
                        snippet["title"],
                        owner_name,
                        parse(snippet["publishedAt"]) if snippet.get("publishedAt") else None,
                        channel_id,
                    )

        return {
            vid: (cached[0], cached[2], cached[3])
            for vid in video_ids
            if (cached := self.cache.video_cache.get(vid)) is not None
        }

    def fetch_video_metadata(
        self,
        youtube_client,
        video_id: str,
    ) -> Tuple[Optional[str], Optional[str], Optional[datetime], Optional[str]]:
        """
        Fetches single video metadata.

        Returns:
            Tuple of (title, owner_name, publish_date, owner_id)
        """
        if video_id in self.cache.video_cache:
            return self.cache.video_cache[video_id]

        def _req(svc):
            return svc.videos().list(
                part="snippet",
                id=video_id,
                maxResults=1,
                fields="items(snippet(channelId,title,publishedAt))",
            )

        resp, service = youtube_client.retry_request(_req)
        if not resp or not resp.get("items"):
            return None, None, None, None

        snip = resp["items"][0]["snippet"]
        owner_id = snip["channelId"]
        title = snip["title"]
        publish_dt = parse(snip["publishedAt"]) if snip.get("publishedAt") else None
        owner_name = self.cache.channel_cache.get(owner_id)

        self.cache.video_cache[video_id] = (title, owner_name, publish_dt, owner_id)
        return title, owner_name, publish_dt, owner_id

In [ ]:


# ==============================================================================
# File: api/comments.py
# ==============================================================================
import logging
from datetime import datetime, timezone
from typing import Dict, List, Optional

from dateutil.parser import parse

from config import CUTOFF_DATE

logger = logging.getLogger(__name__)


class CommentManager:
    """Handles fetching and processing of YouTube comments."""

    def __init__(self, metadata_manager):
        self.metadata = metadata_manager

    def get_most_recent_comment_date(self, db, channel_id: str, video_id: str, fallback_date: str):
        """Get the most recent comment date from database or fallback."""
        row = db.get_most_recent_comment(channel_id, video_id)
        return parse(row["updated_at"]) if row else parse(fallback_date)

    def fetch_comments_page(self, youtube_client, video_id: str, page_token: str, max_results: int):
        """Fetch a single page of comments."""
        def _req(svc):
            return svc.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=max_results,
                order="time",
                pageToken=page_token,
            )

        resp, service = youtube_client.retry_request(_req)
        if not resp:
            return [], None, service
        return resp.get("items", []), resp.get("nextPageToken"), service

    def fetch_comments_with_resume(
        self,
        youtube_client,
        video_id: str,
        channel_id: str,
        db,
        max_results: int = 100,
        initial_fetch_date: str = CUTOFF_DATE,
        ignore_progress: bool = False,
        **metadata_kwargs
    ) -> Dict:
        """
        Fetches comments with resume capability.

        Returns:
            Dict with keys 'comments' and 'youtube_service'
        """
        # Get or fetch metadata
        video_title = metadata_kwargs.get('video_title')
        channel_name = metadata_kwargs.get('channel_name')
        video_publish_date = metadata_kwargs.get('video_publish_date')

        if not all([video_title, channel_name, video_publish_date]):
            video_title, channel_name, video_publish_date, _ = self.metadata.fetch_video_metadata(
                youtube_client, video_id
            )

        if not all([video_title, channel_name, video_publish_date]):
            logger.warning("Skipping %s: incomplete metadata", video_id)
            return {"comments": [], "youtube_service": youtube_client.service}

        # Pagination state
        page_token = None if ignore_progress else db.get_progress(video_id)
        prev_token = "__first_pass"
        most_recent = self.get_most_recent_comment_date(
            db, channel_id, video_id, initial_fetch_date
        )

        results = []

        # Page loop
        while True:
            if page_token is not None and page_token == prev_token:
                logger.warning('Page token "%s" repeated – aborting', page_token)
                break
            prev_token = page_token

            page, next_token, service = self.fetch_comments_page(
                youtube_client, video_id, page_token, max_results
            )
            if not page:
                db.save_progress(video_id, None)
                break

            new_rows = []
            for item in page:
                snip = item["snippet"]["topLevelComment"]["snippet"]
                c_date = parse(snip["updatedAt"])
                if c_date > most_recent and c_date >= video_publish_date:
                    new_rows.append({
                        "video_id": video_id,
                        "video_title": video_title,
                        "channel_id": channel_id,
                        "channel_name": channel_name,
                        "video_publish_date": video_publish_date,
                        "comment_id": item["id"],
                        "author": snip.get("authorDisplayName"),
                        "author_channel_id": snip.get("authorChannelId", {}).get("value"),
                        "text": snip.get("textDisplay"),
                        "like_count": snip.get("likeCount"),
                        "published_at": snip.get("publishedAt"),
                        "updated_at": snip["updatedAt"],
                    })

            if new_rows:
                results.extend(new_rows)

            if next_token:
                db.save_progress(video_id, next_token)
                page_token = next_token
                continue
            else:
                db.save_progress(video_id, None)
                break

        return {"comments": results, "youtube_service": youtube_client.service}


In [ ]:

# ==============================================================================
# File: api/channels.py
# ==============================================================================
import logging
from datetime import datetime, timezone, timedelta
from typing import Dict, List, Optional, Tuple

from dateutil.parser import parse

logger = logging.getLogger(__name__)


class ChannelManager:
    """Handles channel-related operations."""

    def __init__(self, metadata_manager, comment_manager):
        self.metadata = metadata_manager
        self.comments = comment_manager

    def get_top_channels(self, youtube_client, channels: Dict, n: int = 3) -> List[Tuple[str, Dict]]:
        """Get top N channels by subscriber count."""
        for name, info in channels.items():
            if info.get("channel_id"):
                def _req(svc):
                    return svc.channels().list(
                        part="statistics",
                        id=info["channel_id"],
                        maxResults=1,
                    )
            elif info.get("handle"):
                def _req(svc):
                    return svc.channels().list(
                        part="statistics",
                        forHandle=info["handle"].lstrip("@"),
                        maxResults=1,
                    )
            else:
                logger.warning("Channel %s has neither channel_id nor handle", name)
                info["subscriber_count"] = 0
                continue

            resp, service = youtube_client.retry_request(_req)
            try:
                subs = int(
                    resp.get("items", [{}])[0]
                    .get("statistics", {})
                    .get("subscriberCount", "0")
                )
            except (AttributeError, IndexError, ValueError, TypeError):
                logger.warning("Could not fetch subscriber count for %s", name)
                subs = 0

            info["subscriber_count"] = subs

        return sorted(
            channels.items(),
            key=lambda x: x[1].get("subscriber_count", 0),
            reverse=True
        )[:n]

    def get_last_upload_date(self, youtube_client, channel_id: str) -> Optional[datetime]:
        """Get the date of the latest upload for a channel."""
        def _chan_details(svc):
            return svc.channels().list(
                part="contentDetails",
                id=channel_id,
                maxResults=1,
            )

        resp, service = youtube_client.retry_request(_chan_details)
        if not resp or not resp.get("items"):
            return None

        uploads_pl = resp["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]

        def _pl_items(svc):
            return svc.playlistItems().list(
                part="contentDetails",
                playlistId=uploads_pl,
                maxResults=1
            )

        resp, service = youtube_client.retry_request(_pl_items)
        if not resp or not resp.get("items"):
            return None

        last_date = resp["items"][0]["contentDetails"]["videoPublishedAt"]
        return parse(last_date).astimezone(timezone.utc)

    def verify_channels(
        self,
        youtube_client,
        channels: Dict[str, dict],
        cutoff_days: int = 365
    ) -> Dict[str, dict]:
        """Verify channel existence and activity."""
        cutoff_dt = datetime.now(tz=timezone.utc) - timedelta(days=cutoff_days)
        report = {}

        for name, info in channels.items():
            cid = info["channel_id"]

            def _exists(svc):
                return svc.channels().list(part="id", id=cid, maxResults=1)

            resp, service = youtube_client.retry_request(_exists)
            exists = bool(resp and resp.get("items"))

            if not exists:
                report[name] = {"exists": False, "last_upload": None, "inactive": True}
                continue

            last_dt = self.get_last_upload_date(youtube_client, cid)
            inactive = (last_dt is None) or (last_dt < cutoff_dt)

            report[name] = {
                "exists": True,
                "last_upload": last_dt,
                "inactive": inactive,
            }

        return report


In [ ]:

# ==============================================================================
# File: api/playlists.py
# ==============================================================================
import json
import logging
import os
import pathlib
from typing import List, Tuple

logger = logging.getLogger(__name__)


class PlaylistManager:
    """Handles playlist operations."""

    def generate_videos_by_search(self, youtube_client, channel_id: str, keyword: str, max_results: int = 50):
        """Generate videos by searching within a channel."""
        page_token = None
        while True:
            def _req(svc):
                return svc.search().list(
                    part="id",
                    channelId=channel_id,
                    q=keyword,
                    type="video",
                    maxResults=max_results,
                    order="date",
                    pageToken=page_token,
                )

            resp, service = youtube_client.retry_request(_req)
            if not resp:
                break

            for item in resp.get("items", []):
                vid = item["id"]["videoId"]
                # You'll need to inject metadata_manager here
                # v_title, ch_name, pub_dt, _ = metadata_manager.fetch_video_metadata(youtube_client, vid)
                # if not all([v_title, ch_name, pub_dt]):
                #     continue
                # yield vid, service, v_title, ch_name, pub_dt
                yield vid, service

            page_token = resp.get("nextPageToken")
            if not page_token:
                break

    def generate_playlists(self, youtube_client, channel_id: str, keywords: List[str], max_results: int = 10):
        """Generate playlists matching keywords."""
        page_token = None
        while True:
            def _req(svc):
                return svc.playlists().list(
                    part="id,snippet",
                    channelId=channel_id,
                    maxResults=max_results,
                    pageToken=page_token,
                )

            resp, service = youtube_client.retry_request(_req)
            if not resp:
                break

            for item in resp.get("items", []):
                title = item["snippet"]["title"].lower()
                if any(k.lower() in title for k in keywords):
                    yield item["id"], item["snippet"]["title"]

            page_token = resp.get("nextPageToken")
            if not page_token:
                break

    def generate_videos(self, youtube_client, playlist_id: str, max_results: int = 50):
        """Generate video IDs from a playlist."""
        page_token = None
        while True:
            def _req(svc):
                return svc.playlistItems().list(
                    part="snippet,contentDetails",
                    playlistId=playlist_id,
                    maxResults=max_results,
                    pageToken=page_token,
                )

            resp, service = youtube_client.retry_request(_req)
            if not resp:
                break

            for item in resp.get("items", []):
                yield item["contentDetails"]["videoId"]

            page_token = resp.get("nextPageToken")
            if not page_token:
                break

    def cached_search_playlists(self, youtube_client, channel_id: str, keywords: List[str]) -> List[Tuple[str, str]]:
        """Cache and retrieve playlists based on keywords."""
        cache_file = f"yt_cache/pl_{channel_id}.json"
        if os.path.exists(cache_file):
            with open(cache_file, "r", encoding="utf-8") as fp:
                return json.load(fp)

        playlists = list(self.generate_playlists(youtube_client, channel_id, keywords))

        if not playlists:
            def _search_req(svc):
                return svc.search().list(
                    part="snippet",
                    channelId=channel_id,
                    q=" | ".join(keywords),
                    type="playlist",
                    maxResults=5,
                    fields="items(id/playlistId,snippet/title)",
                )

            resp, service = youtube_client.retry_request(_search_req)
            if resp:
                playlists = [
                    (it["id"]["playlistId"], it["snippet"]["title"])
                    for it in resp.get("items", [])
                ]

        pathlib.Path("yt_cache").mkdir(exist_ok=True)
        with open(cache_file, "w", encoding="utf-8") as fp:
            json.dump(playlists, fp)

        return playlists


In [ ]:

# ==============================================================================
# File: core/processor.py
# ==============================================================================
import logging
from typing import Dict, List

from config import CHANNELS, KEYWORDS, CUTOFF_DATE

logger = logging.getLogger(__name__)


class YouTubeProcessor:
    """Main processor that orchestrates all YouTube operations."""

    def __init__(self, youtube_client, metadata_manager, comment_manager,
                 channel_manager, playlist_manager):
        self.youtube_client = youtube_client
        self.metadata = metadata_manager
        self.comments = comment_manager
        self.channels = channel_manager
        self.playlists = playlist_manager

    def get_comments_by_playlist(self, channel_id: str, db, keywords: List[str], max_results: int = 5):
        """Fetch comments from playlist videos."""
        processed = set()
        playlists = self.playlists.cached_search_playlists(self.youtube_client, channel_id, keywords)

        if not playlists:
            logger.info("No matching playlists for %s", channel_id)
            return

        for pl_id, pl_title in playlists:
            logger.info("Processing playlist %s – %s", pl_id, pl_title)
            try:
                for vid in self.playlists.generate_videos(self.youtube_client, pl_id, max_results=100):
                    if vid in processed:
                        continue
                    processed.add(vid)

                    res = self.comments.fetch_comments_with_resume(
                        self.youtube_client, vid, channel_id, db,
                        max_results=100, initial_fetch_date=CUTOFF_DATE
                    )
                    if res["comments"]:
                        db.insert_comments(res["comments"])
            except Exception as exc:
                logger.error("Error processing playlist %s: %s", pl_id, exc)

    def get_all_channel_comments(self, channel_id: str, db, max_results: int = 100, cutoff_date=CUTOFF_DATE):
        """Fetch all comments from a channel."""
        from datetime import datetime, timezone
        from dateutil.parser import parse

        cutoff_dt = cutoff_date if isinstance(cutoff_date, datetime) else parse(cutoff_date)
        if cutoff_dt.tzinfo is None:
            cutoff_dt = cutoff_dt.replace(tzinfo=timezone.utc)

        progress_key = f"chan::{channel_id}"
        page_token = db.get_progress(progress_key)
        if page_token is None and db.progress_exists(progress_key):
            logger.debug("Channel %s up-to-date; skipping", channel_id)
            return

        # Get channel name
        def _chan_req(svc):
            return svc.channels().list(part="snippet", id=channel_id, maxResults=1)

        c_resp, service = self.youtube_client.retry_request(_chan_req)
        if not c_resp or not c_resp.get("items"):
            logger.warning("Channel %s not found", channel_id)
            return

        chan_name = c_resp["items"][0]["snippet"]["title"]

        # Main page loop
        while True:
            def _page_req(svc):
                return svc.commentThreads().list(
                    part="snippet",
                    allThreadsRelatedToChannelId=channel_id,
                    maxResults=max_results,
                    order="time",
                    pageToken=page_token,
                )

            resp, service = self.youtube_client.retry_request(_page_req)
            if not resp or not resp.get("items"):
                db.save_progress(progress_key, None)
                break

            vids = {
                itm["snippet"]["topLevelComment"]["snippet"].get("videoId")
                for itm in resp["items"]
            }
            vids.discard(None)
            meta = self.metadata.batch_fetch_video_metadata(self.youtube_client, list(vids))

            rows = []
            for itm in resp["items"]:
                snip = itm["snippet"]["topLevelComment"]["snippet"]
                comment_dt = parse(snip["updatedAt"]).astimezone(timezone.utc)
                if comment_dt < cutoff_dt:
                    db.save_progress(progress_key, None)
                    return

                vid = snip.get("videoId")
                if not vid or vid not in meta:
                    continue

                title, pub_dt, owner_id = meta[vid]
                if owner_id != channel_id:
                    continue

                rows.append({
                    "video_id": vid,
                    "video_title": title,
                    "channel_id": channel_id,
                    "channel_name": chan_name,
                    "video_publish_date": pub_dt,
                    "comment_id": itm["id"],
                    "author": snip.get("authorDisplayName"),
                    "author_channel_id": snip.get("authorChannelId"),
                    "text": snip.get("textOriginal"),
                    "like_count": snip.get("likeCount"),
                    "published_at": snip.get("publishedAt"),
                    "updated_at": snip["updatedAt"],
                })

            if rows:
                db.insert_comments(rows)

            page_token = resp.get("nextPageToken")
            db.save_progress(progress_key, page_token)
            if not page_token:
                break

    def process_channels(self, db, channels: Dict = None, keywords: List[str] = None):
        """Process a list of channels."""
        channels = channels or CHANNELS
        keywords = keywords if isinstance(keywords, list) else list(keywords or KEYWORDS)

        for name, info in channels.items():
            cid = info["channel_id"]
            if info.get("only_wow"):
                self.get_all_channel_comments(cid, db, max_results=100)
            else:
                self.get_comments_by_playlist(cid, db, keywords=keywords, max_results=100)


In [ ]:

# ==============================================================================
# File: cli/arguments.py
# ==============================================================================
import argparse


def parse_cli() -> argparse.Namespace:
    """Parse command-line arguments."""
    p = argparse.ArgumentParser(
        description="YouTube comment scraper with flexible channel selection"
    )
    p.add_argument(
        "--channels",
        metavar="NAME1,NAME2",
        help="Comma-separated list of channel names to include (keys in CHANNELS).",
    )
    p.add_argument(
        "--skip",
        metavar="NAME1,NAME2",
        help="Comma-separated list of channel names to exclude.",
    )
    p.add_argument(
        "--types",
        metavar="TAG1,TAG2",
        help="Include only channels whose `tags` contain ANY of these labels.",
    )
    p.add_argument(
        "--exclude-types",
        metavar="TAG1,TAG2",
        help="Exclude channels whose `tags` contain ANY of these labels "
             "(applied after --types / --channels).",
    )
    p.add_argument(
        "--limit-top",
        type=int,
        metavar="N",
        help="Keep only the top-N channels by subscriber count "
             "(after previous filters, before --skip / --exclude-types).",
    )
    p.add_argument(
        "--max-inactive-days",
        type=int,
        metavar="DAYS",
        help="Skip any channel whose last upload is older than DAYS.",
    )
    p.add_argument(
        "--verify-channels",
        action="store_true",
        help="Just print a health report for every channel and exit.",
    )
    p.add_argument(
        "--verify-cutoff-days",
        type=int,
        default=365,
        metavar="N",
        help="Channel is flagged inactive if no upload in the last N days "
             "(used with --verify-channels).",
    )

    return p.parse_args()


In [ ]:

# ==============================================================================
# File: cli/channel_filter.py
# ==============================================================================
import logging
from datetime import datetime, timezone, timedelta
from typing import Dict

logger = logging.getLogger(__name__)


class ChannelFilter:
    """Handles filtering of channels based on various criteria."""

    def __init__(self, channel_manager):
        self.channel_manager = channel_manager

    def apply_filters(self, channels: Dict, args, youtube_client) -> Dict:
        """Apply all filters based on command line arguments."""
        selected = channels.copy()

        # Remove outdated channels
        selected = {k: v for k, v in selected.items() if not v.get("outdated")}

        # Filter by specific channel names
        if args.channels:
            keep = {n.strip() for n in args.channels.split(",")}
            selected = {k: v for k, v in selected.items() if k in keep}

        # Filter by channel tags
        if args.types:
            tags_wanted = {t.strip().lower() for t in args.types.split(",")}
            selected = {
                k: v for k, v in selected.items()
                if any(t.lower() in tags_wanted for t in v.get("tags", []))
            }

        # Limit to top N channels by subscriber count
        if args.limit_top:
            selected = dict(
                self.channel_manager.get_top_channels(youtube_client, selected, args.limit_top)
            )

        # Skip specific channels
        if args.skip:
            skip_set = {n.strip() for n in args.skip.split(",")}
            selected = {k: v for k, v in selected.items() if k not in skip_set}

        # Exclude channels by tags
        if args.exclude_types:
            bad = {t.strip().lower() for t in args.exclude_types.split(",")}
            selected = {
                k: v for k, v in selected.items()
                if not any(t.lower() in bad for t in v.get("tags", []))
            }

        # Filter by channel activity
        if args.max_inactive_days:
            cutoff = datetime.now(tz=timezone.utc) - timedelta(days=args.max_inactive_days)
            still_active = {}
            for name, info in selected.items():
                last_dt = self.channel_manager.get_last_upload_date(youtube_client, info["channel_id"])
                if last_dt and last_dt >= cutoff:
                    still_active[name] = info
                else:
                    logger.info(
                        "Skipping %s (last upload %s)",
                        name,
                        last_dt.date() if last_dt else "never",
                    )
            selected = still_active

        return selected


In [ ]:

# ==============================================================================
# File: main.py
# ==============================================================================
import logging
import sys
from pprint import pprint

from utils.logging_setup import setup_logging
from utils.cache import CacheManager
from api.youtube_client import YouTubeClient
from api.metadata import MetadataManager
from api.comments import CommentManager
from api.channels import ChannelManager
from api.playlists import PlaylistManager
from core.processor import YouTubeProcessor
from cli.arguments import parse_cli
from cli.channel_filter import ChannelFilter
from config import CHANNELS
from database_con import DatabaseConnection

# Initialize logging
setup_logging()
logger = logging.getLogger(__name__)


def main() -> None:
    """Main entry point for the YouTube comment scraper."""
    try:
        args = parse_cli()

        # Initialize components
        cache_manager = CacheManager()
        youtube_client = YouTubeClient()
        metadata_manager = MetadataManager(cache_manager)
        comment_manager = CommentManager(metadata_manager)
        channel_manager = ChannelManager(metadata_manager, comment_manager)
        playlist_manager = PlaylistManager()

        processor = YouTubeProcessor(
            youtube_client, metadata_manager, comment_manager,
            channel_manager, playlist_manager
        )

        channel_filter = ChannelFilter(channel_manager)

        # Apply filters to channels
        selected_channels = channel_filter.apply_filters(CHANNELS, args, youtube_client)

        if not selected_channels:
            logger.error("No channels left to process after filtering.")
            return

        # Handle verification mode
        if args.verify_channels:
            report = channel_manager.verify_channels(
                youtube_client, selected_channels, cutoff_days=args.verify_cutoff_days
            )

            # Pretty print the report
            formatted_report = {
                k: {
                    "exists": v["exists"],
                    "inactive": v["inactive"],
                    "last_upload": (
                        v["last_upload"].date().isoformat() if v["last_upload"] else None
                    ),
                } for k, v in report.items()
            }
            pprint(formatted_report)
            return

        # Process channels
        with DatabaseConnection() as db:
            processor.process_channels(db, channels=selected_channels)

    except KeyboardInterrupt:
        logger.info("Process interrupted by user. Exiting gracefully.")
        sys.exit(0)
    except Exception as e:
        logger.error("Unexpected error: %s", e, exc_info=True)
        sys.exit(1)


if __name__ == "__main__":
    main()


In [ ]:

# ==============================================================================
# File: __init__.py (for each package)
# ==============================================================================

# utils/__init__.py
"""Utility modules for the YouTube comment scraper."""

# api/__init__.py
"""API interaction modules for YouTube Data API."""

# core/__init__.py
"""Core processing logic for the YouTube comment scraper."""

# cli/__init__.py
"""Command-line interface modules."""


In [ ]:

# ==============================================================================
# File: requirements.txt
# ==============================================================================
"""
google-api-python-client>=2.0.0
python-dateutil>=2.8.0
"""


In [ ]:

# ==============================================================================
# File: README.md
# ==============================================================================
"""
# YouTube Comment Scraper - Modular Version

## Overview
A modular YouTube comment scraper that efficiently fetches comments from YouTube channels and playlists with advanced filtering, caching, and retry mechanisms.

## Structure

### Core Modules

#### `utils/`
- **logging_setup.py**: Configures logging system
- **cache.py**: LRU cache implementation for video/channel metadata

#### `api/`
- **youtube_client.py**: YouTube API client with quota management and retries
- **metadata.py**: Video and channel metadata fetching
- **comments.py**: Comment fetching and processing
- **channels.py**: Channel operations (verification, stats, etc.)
- **playlists.py**: Playlist operations and video discovery

#### `core/`
- **processor.py**: Main orchestration logic

#### `cli/`
- **arguments.py**: Command-line argument parsing
- **channel_filter.py**: Channel filtering logic

## Features

### Modular Design
- **Separation of Concerns**: Each module has a specific responsibility
- **Dependency Injection**: Components are injected rather than tightly coupled
- **Easy Testing**: Each module can be tested independently
- **Extensibility**: New features can be added without modifying existing code

### API Management
- **Quota Handling**: Automatic API key rotation when quotas are exhausted
- **Retry Logic**: Exponential backoff for transient errors
- **Rate Limiting**: Respect API rate limits

### Caching
- **LRU Cache**: Efficient memory usage with automatic eviction
- **Persistent Storage**: Cache survives across runs
- **Metadata Caching**: Avoid redundant API calls for video/channel info

### Resume Capability
- **Progress Tracking**: Resume from where processing left off
- **Database Integration**: Persistent progress storage
- **Incremental Updates**: Only fetch new comments

## Usage

```bash
# Basic usage
python main.py

# Filter by channel types
python main.py --types "tech,gaming"

# Verify channel health
python main.py --verify-channels

# Process specific channels
python main.py --channels "channel1,channel2"

# Exclude inactive channels
python main.py --max-inactive-days 30
```

## Configuration

Create a `config.py` file with:
```python
API_KEYS = ["your_api_key_1", "your_api_key_2"]
CHANNELS = {
    "channel_name": {
        "channel_id": "UC...",
        "tags": ["tech", "tutorials"],
        "only_wow": False
    }
}
KEYWORDS = ["python", "programming", "tutorial"]
CUTOFF_DATE = "2023-01-01T00:00:00Z"
LOG_CONFIG_PATH = "logging_config.json"
```

## Benefits of Modular Design

1. **Maintainability**: Easier to understand, modify, and debug
2. **Testability**: Each component can be unit tested in isolation
3. **Reusability**: Modules can be reused in other projects
4. **Scalability**: Easy to add new features or modify existing ones
5. **Collaboration**: Multiple developers can work on different modules
6. **Error Isolation**: Issues in one module don't affect others

## Migration from Original Script

The modular version maintains the same functionality while providing:
- Better code organization
- Improved error handling
- Easier configuration management
- Enhanced logging capabilities
- More flexible component composition

All original features are preserved, including:
- Channel verification
- Comment fetching with resume
- Playlist processing
- Subscriber-based filtering
- Activity-based filtering
"""